# 06 — Model Selection, Stability, and Explainability

This notebook focuses on judgement: how stable is the structure we discovered?


In [ ]:
import sys
from pathlib import Path

from sklearn.cluster import KMeans

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_customer_segmentation_data
from unsup_lab.evaluation import evaluate_k_range
from unsup_lab.preprocessing import scale_features
from unsup_lab.reporting import cluster_profile
from unsup_lab.stability import (
    bootstrap_cluster_stability,
    outlier_sensitivity,
    pairwise_adjusted_mutual_information,
    repeated_run_labels,
    scaling_sensitivity,
    stability_report,
)

In [ ]:
dataset = make_customer_segmentation_data(n_customers=1_500, random_state=123)
features = dataset.features
scaled = scale_features(features, method="standard")

# A factory maps a random seed to a fresh estimator, so the stability
# tools can re-fit the same configuration under different randomness.
def kmeans_factory(seed: int) -> KMeans:
    return KMeans(n_clusters=5, n_init=10, random_state=seed)

## Internal metrics across k

In [ ]:
metrics = evaluate_k_range(
    scaled.to_numpy(),
    estimator_factory=lambda k: KMeans(n_clusters=k, n_init=20, random_state=123),
    k_values=list(range(2, 11)),
)

metrics.round(3)

## Seed stability

In [ ]:
# Re-fit KMeans under 20 seeds and measure how much the partitions agree.
seed_runs = repeated_run_labels(
    scaled.to_numpy(), kmeans_factory, n_runs=20, random_state=0
)
seed_stability = pairwise_adjusted_mutual_information(seed_runs)
seed_stability

## Bootstrap stability

Seed agreement only varies the initialisation. Bootstrap stability also varies *which customers are present*, which is a stronger test: if the segments survive resampling, they are unlikely to be an artefact of a few rows.

In [ ]:
bootstrap = bootstrap_cluster_stability(
    scaled.to_numpy(), kmeans_factory, n_bootstrap=20, sample_fraction=0.8, random_state=0
)
bootstrap

## Sensitivity to scaling and outliers

Two more failure modes worth checking before trusting a segmentation: does the partition depend on the scaling choice, and does a handful of injected outliers reshuffle the assignments?

In [ ]:
scaling = scaling_sensitivity(features, kmeans_factory, random_state=0)
scaling.round(3)

In [ ]:
outliers = outlier_sensitivity(
    scaled.to_numpy(), kmeans_factory,
    contamination_levels=(0.0, 0.02, 0.05, 0.1), random_state=0,
)
outliers.round(3)

## Explain the selected clustering

In [ ]:
selected = kmeans_factory(123).fit_predict(scaled.to_numpy())
profile = cluster_profile(features, selected)

profile.round(2)

## Automated stability report

`stability_report` bundles every diagnostic above into a single Markdown summary that can be saved as an artefact or pasted into a review.

In [ ]:
from IPython.display import Markdown, display

report = stability_report(scaled, kmeans_factory, n_runs=10, n_bootstrap=10, random_state=0)
display(Markdown(report))

## Interpretation

A good unsupervised learning workflow should ask:

- Does the result change under another random seed?
- Does it change under another scaling method?
- Does it change when outliers are removed?
- Are the clusters actionable?
- Are the clusters stable enough to support decisions?

These questions are often more important than the choice of algorithm.
